# Pages You May Like

Suggests pages to a user based on pages their friends have liked, ranked by how many friends liked each page. Uses the same social graph dataset as `people_you_may_know.ipynb`.

In [ ]:
import json

def load_data(filename):
    with open(filename, 'r') as f:
        return json.load(f)

In [ ]:
def find_pages_you_might_like(user_id, data):
    # Map each user id -> set of page ids they've liked
    user_pages = {user['id']: set(user['liked_pages']) for user in data['users']}

    if user_id not in user_pages:
        return []

    my_pages = user_pages[user_id]

    # Find this user's friend list
    friends = []
    for user in data['users']:
        if user['id'] == user_id:
            friends = user['friends']
            break

    # Count how many friends like each page the user doesn't already like
    suggestions = {}
    for friend_id in friends:
        for page_id in user_pages.get(friend_id, set()):
            if page_id not in my_pages:
                suggestions[page_id] = suggestions.get(page_id, 0) + 1

    # Rank by number of friends who liked the page, most-liked first
    sorted_suggestions = sorted(suggestions.items(), key=lambda x: x[1], reverse=True)
    return [page_id for page_id, _ in sorted_suggestions]

In [ ]:
data = load_data("data/massive_data.json")
user_id = 1
recommended_page_ids = find_pages_you_might_like(user_id, data)

page_names = {p['id']: p['name'] for p in data['pages']}
recommended_names = [page_names[pid] for pid in recommended_page_ids]

print(f"Recommended page ids for user {user_id}:", recommended_page_ids)
print(f"Recommended pages for user {user_id}:", recommended_names)